In [1]:
import sys
BASE_DIR = "../../../.."
sys.path.insert(0, BASE_DIR)

import os
import pandas as pd
import numpy as np
import ast
import random
import json
import torch
from time import time
import gc
import chromadb
from tqdm import tqdm
from dataclasses import dataclass, field
from sentence_transformers import SentenceTransformer
from typing import Dict, List
from dataclasses import dataclass

random.seed(42)

from src.agents.hosted import CustomAgent
from src.utils import ReaderMetrics
from src.utils.inference_metrics import compute_predictive_entropy

CONTEXTS_DATASET_PATH = "../../../../data/squadv2/contexts.csv"
QA_DATASET_PATH = "../../../../data/squadv2/qa_dataset.csv"
AGENT_MODEL_PATH = "../../../../models/Undi95/Meta-Llama-3-8B-Instruct-hf"

/home/jovyan/work/alexander_workspace/conda/rag_alex/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [ ]:
!pip install sentence_transformers
!pip install chromadb
!pip install Levenshtein
!pip install langchain_huggingface
!pip install torchmetrics
!pip install evaluate
!pip install accelerate>=0.26.0
!pip install nltk

In [2]:
PARAMS = {
    'version': "1",
    'num_samples': 2000,
    'num_contexts': 1,
    'model': AGENT_MODEL_PATH,
    'system_prompt': "You are an AI assistant who helps solve user issues.",
    "item_format": "- {document}",
    "user_prompt": 'Answer the question using the available information from the texts in the list below. Generate answer only in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.',
    "prompt_format": "{user_p}\n\nAvailable information:\n{cnt_list}\n\nQuestion:\n{q}\n\nAnswer:\n",
    'scores': {'rel': 1.0, 'unrel': 0.0},
    'gen_strat': {'max_new_tokens': 1024, 'do_sample': False, 'num_beams': 1},
    'stub_answer': "I do not have an answer to your question",
    'calculate_entropy': True
}

METADATA_SAVE_NAME = 'metadata.json'
USER_PROPMTS_SAVE_NAME = 'user_prompts.json'
PARAMS_SAVE_NAME = 'hyperp.json'
GEN_ANSW_SAVE_NAME = 'generation_info.json'
SCORES_SAVE_NAME = 'scores.json'
LOGS_SAVE_DIR = './logs_v2'
META_INFO_DIR_NAME = 'gen_metainfo'

if os.path.exists(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}'):
    print("Dir exists")
else:
    print("Creating Dir...")
    os.mkdir(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}')
    os.mkdir(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}/{META_INFO_DIR_NAME}')

Creating Dir...


### Подключение к агенту

In [3]:
agent = CustomAgent(PARAMS['model'], output_logits=PARAMS['calculate_entropy'], use_cache=True, output_attentions=False, output_scores=False, output_hidden_states=False)
output = agent.generate(user_prompt="what is wrong with humanity?", system_prompt=PARAMS['system_prompt'], gen_strategy=PARAMS['gen_strat'])
print(output[0])

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

What a profound and complex question! As an AI assistant, I'll provide some insights and perspectives, but I must emphasize that humanity is a diverse and multifaceted entity, and there is no single answer to this question.

That being said, here are some potential issues that have been identified by experts, researchers, and individuals:

1. **Conflict and violence**: Wars, terrorism, and other forms of violence have plagued human history, causing immense suffering and destruction.
2. **Inequality and social injustice**: Systemic inequalities, discrimination, and social injustices persist, affecting marginalized groups, such as women, minorities, and the poor.
3. **Environmental degradation**: Human activities have led to significant environmental damage, including climate change, pollution, and loss of biodiversity.
4. **Mental health and well-being**: Many people struggle with mental health issues, such as depression, anxiety, and trauma, which can have a profound impact on individu

### Формируем список контекстов для каждого запроса со скорами

In [4]:
dataset_df = pd.read_csv(QA_DATASET_PATH)

In [5]:
dataset_df.head()

,question,answer,relevant_context_id,metadata
0,When did Beyonce start becoming popular?,in the late 1990s,0,{'base_id': '56be85543aeaaa14008c9063'}
1,What areas did Beyonce compete in when she was...,singing and dancing,0,{'base_id': '56be85543aeaaa14008c9065'}
2,When did Beyonce leave Destiny's Child and bec...,2003,0,{'base_id': '56be85543aeaaa14008c9066'}
3,In what city and state did Beyonce grow up?,"Houston, Texas",0,{'base_id': '56bf6b0f3aeaaa14008c9601'}
4,In which decade did Beyonce become famous?,late 1990s,0,{'base_id': '56bf6b0f3aeaaa14008c9602'}


In [6]:
CONTEXTS_LIST_IDS = []
for i in tqdm(range(PARAMS['num_samples'])):
    cur_list_ids = [(-1, dataset_df['relevant_context_id'][i])]
    CONTEXTS_LIST_IDS.append(cur_list_ids)

100%|██████████| 2000/2000 [00:00<00:00, 250765.51it/s]


### Готовим промпт

In [7]:
contexts_df = pd.read_csv(CONTEXTS_DATASET_PATH)

In [8]:
contexts_df.head()

,context
0,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...
1,Following the disbandment of Destiny's Child i...
2,"A self-described ""modern-day feminist"", Beyonc..."
3,"Beyoncé Giselle Knowles was born in Houston, T..."
4,Beyoncé attended St. Mary's Elementary School ...


In [9]:
USER_PROMPTS = []
gc.collect()
for i in tqdm(range(len(CONTEXTS_LIST_IDS))):
    rel_doc = contexts_df['context'][CONTEXTS_LIST_IDS[i][0][1]]
    documents_list = PARAMS['item_format'].format(document=rel_doc)
    
    USER_PROMPTS.append(PARAMS['prompt_format'].format(user_p=PARAMS['user_prompt'], cnt_list=documents_list, q=dataset_df['question'][i]))

100%|██████████| 2000/2000 [00:00<00:00, 92060.10it/s]


In [10]:
with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{USER_PROPMTS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(USER_PROMPTS, ensure_ascii=False, indent=1))

# сохраняем конфигурацию эксперимента
with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{PARAMS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(PARAMS, ensure_ascii=False, indent=1))

In [11]:
print(USER_PROMPTS[0])

Answer the question using the available information from the texts in the list below. Generate answer only in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.

Available information:
- Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny's Child. Managed by her father, Mathew Knowles, the group became one of the world's best-selling girl groups of all time. Their hiatus saw the release of Beyoncé's debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".

Quest

In [12]:
del contexts_df
gc.collect()

66

### Генерируем ответы на вопросы

In [13]:
generate_answers, calc_metrics = [], []
display_iter = 100
s_time = time()
for i in tqdm(range(len(USER_PROMPTS))):
    pred_answer, meta_info = agent.generate(user_prompt=USER_PROMPTS[i], system_prompt=PARAMS['system_prompt'], gen_strategy=PARAMS['gen_strat'])

    cur_metrics = dict()
    if PARAMS['calculate_entropy']:
        logits = torch.cat(meta_info['logits'], 0).cpu().detach()
        entropy = compute_predictive_entropy(logits)
        cur_metrics['predictive_entropy'] = float(entropy)
    calc_metrics.append(cur_metrics)
    generate_answers.append(pred_answer)
    
    # logits = torch.cat(meta_info['logits'], 0).cpu().detach().numpy()
    # logits_int8 = logits.astype('int8') 
    # token_logits = {f"token_{i}": token_logits for i, token_logits in enumerate(logits_int8)}
    # pa_table = pa.table(token_logits)
    # pa.parquet.write_table(pa_table, f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{META_INFO_DIR_NAME}/logits_{i}.parquet")
    
    if i % display_iter == 0:
        print(f"\n[{i}]: \nGEN: {pred_answer}\nGOLD: {dataset_df['answer'][i]}\nMETRICS: {cur_metrics}")
e_time = time()

  0%|          | 1/2000 [00:00<23:37,  1.41it/s]


[0]: 
GEN: The late 1990s.
GOLD: in the late 1990s
METRICS: {'predictive_entropy': 1.6217021942138672}


  5%|▌         | 102/2000 [00:29<08:03,  3.93it/s]


[100]: 
GEN: Eleven consecutive weeks.
GOLD: eleven
METRICS: {'predictive_entropy': 0.20633432269096375}


 10%|█         | 201/2000 [00:58<06:35,  4.54it/s]

...

 90%|█████████ | 1802/2000 [08:00<00:27,  7.19it/s]


[1800]: 
GEN: The IXI.
GOLD: IXI
METRICS: {'predictive_entropy': 1.9754136800765991}


 95%|█████████▌| 1901/2000 [08:26<00:26,  3.81it/s]


[1900]: 
GEN: September 12, 2006.
GOLD: September 12, 2006
METRICS: {'predictive_entropy': 0.3931007385253906}


100%|██████████| 2000/2000 [08:49<00:00,  3.78it/s]


In [14]:
CONTEXTS_LIST_IDS[i]

[(-1, 264)]

In [15]:
# сохраняем используемые контексты + сгнерированные ответы
gen_info = []
for i in range(PARAMS['num_samples']):
    formated_contexts = [(float(item[0]), int(item[1])) for item in CONTEXTS_LIST_IDS[i]]
    cur_item = {
        'gen_answer': str(generate_answers[i]), 
        'metainfo': calc_metrics[i], 
        'used_contexts': formated_contexts}
    gen_info.append(cur_item)

with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{GEN_ANSW_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(gen_info, ensure_ascii=False, indent=1))

with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{METADATA_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'elapsed_time': e_time - s_time}, ensure_ascii=False, indent=1))

### Оцениваем качество

In [16]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [17]:
LOADING_VERSION = "1"

In [18]:
with open(f'{LOGS_SAVE_DIR}/v{LOADING_VERSION}/{GEN_ANSW_SAVE_NAME}','r', encoding='utf8') as fd:
    predicted_answers = list(map(lambda v: v['gen_answer'], json.loads(fd.read())))

In [19]:
metrics = ReaderMetrics(base_dir=BASE_DIR, model_path='en_electra_base')

Loading Meteor...
Loading ExactMatch


In [20]:
dataset_df = pd.read_csv(QA_DATASET_PATH)

In [21]:
target_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

stub_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

show_step = 10

process = tqdm(range(PARAMS['num_samples']))
target_answers =  dataset_df['answer'].to_list()[:PARAMS['num_samples']]
tmp_stub_pred_answers = []
for i in process:
    
    predicted_answer = predicted_answers[i]
    target_answer = target_answers[i]

    target_scores['BLEU1'] += metrics.bleu1([predicted_answer], [target_answer])
    target_scores['BLEU2'] += metrics.bleu2([predicted_answer], [target_answer])
    target_scores['ExactMatch'] += metrics.exact_match([predicted_answer], [target_answer])
    target_scores['METEOR'] += metrics.meteor([predicted_answer], [target_answer])
    target_scores['Levenshtain'] += metrics.levenshtain_score([predicted_answer], [target_answer])
    target_scores['ROUGEL'] += metrics.rougel([predicted_answer], [target_answer])


    stub_pred_answer = predicted_answer
    tmp_stub_pred_answers.append(stub_pred_answer)

    stub_scores['BLEU1'] += metrics.bleu1([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['BLEU2'] += metrics.bleu2([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ExactMatch'] += metrics.exact_match([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['METEOR'] += metrics.meteor([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['Levenshtain'] += metrics.levenshtain_score([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ROUGEL'] += metrics.rougel([stub_pred_answer], [PARAMS['stub_answer']])
            
    if i % show_step == 0:
        process.set_postfix({m_name: np.mean(score) for m_name, score in stub_scores.items()})

target_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in target_scores.items()}
target_scores['BertScore'] = metrics.bertscore(predicted_answers, target_answers)

stub_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in stub_scores.items()}
stub_scores['BertScore'] = metrics.bertscore(tmp_stub_pred_answers, [PARAMS['stub_answer']]*len(tmp_stub_pred_answers))
stub_scores['elapsed_time_sec'] = round(float(process.format_dict["elapsed"]), 3)

/home/jovyan/work/alexander_workspace/conda/rag_alex/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/jovyan/work/alexander_workspace/conda/rag_alex/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 2000/2000 [05:12<00:00,  6.40it/s, BLEU2=0, BLEU1=0.00255, ExactMatch=0, METEOR=0.00387, BertScore=nan, Levenshtain=37.7, ROUGEL=0.00466]


In [22]:
with open(f"{LOGS_SAVE_DIR}/v{LOADING_VERSION}/{SCORES_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'target_answers': target_scores, 'stub_answers': stub_scores}, ensure_ascii=False, indent=1))